# 00 — Schema explore

Read-only exploration of the prepared DB source once available.

1. Load `.env` / engine from `db.connection`
2. `INFORMATION_SCHEMA.COLUMNS` for the chosen table/view
3. Fill `fraud_guard.features.ColumnMapping`
4. Do **not** write to the database

In [ ]:
## Шаг A — путь + подключение

In [ ]:
from pathlib import Path
import sys
from sqlalchemy import text

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT), str(ROOT / "src")]

from db.config import get_settings
get_settings.cache_clear()

from db.connection import get_engine

engine = get_engine()
VIEW = "dbo.TargetsByMetrics_RateGetAnswers"

with engine.connect() as conn:
    db = conn.execute(text("SELECT DB_NAME()")).scalar()
    n = conn.execute(text(f"SELECT COUNT_BIG(*) FROM {VIEW}")).scalar()

print("DB:", db)
print("rows:", n)

In [ ]:
## Шаг B — загрузить таблицу в pandas (без печати PII)

In [ ]:
import pandas as pd

df = pd.read_sql(f"SELECT * FROM {VIEW}", engine)
print(df.shape)
print(df.columns.tolist())
df.dtypes

In [ ]:
df[["PrintStore", "Year", "Month", "Question_ID", "Answer_Value", "PhoneFromLog"]].head()

In [ ]:
## Шаг C — период и объём

In [ ]:
print(df["AnswerTime"].min(), df["AnswerTime"].max())
print(df.groupby(["Year", "Month"]).size())

In [ ]:
print("stores:", df["PrintStore"].nunique())
print("PrintDateTime range:", df["PrintDateTime"].min(), "→", df["PrintDateTime"].max())

In [ ]:
## добить ясность (опционально, до D)

In [ ]:
print("unique ParticipateNumber:", df["ParticipateNumber"].nunique())
print("unique PrintStore+Pos+Order+PrintDateTime:", 
      df.groupby(["PrintStore","PrintPos","PrintOrder","PrintDateTime"]).ngroups)
print("rows per participation: min/median/max =",
      df.groupby("ParticipateNumber").size().agg(["min","median","max"]).to_dict())
print("AnswerDate nunique:", df["AnswerDate"].nunique())
print("AnswerDate value_counts:\n", df["AnswerDate"].value_counts())

In [ ]:
## точно убедиться

In [ ]:
# 1) Календарные даты ответа
print("AnswerDate unique:", sorted(df["AnswerDate"].dropna().dt.normalize().unique()))
print("nunique AnswerDate:", df["AnswerDate"].dt.normalize().nunique())

# 2) Календарные даты времени ответа
print("AnswerTime dates unique:", sorted(df["AnswerTime"].dropna().dt.date.unique()))
print("nunique AnswerTime.date:", df["AnswerTime"].dt.date.nunique())

# 3) Бизнес-дата заказа (PrintDate)
print("PrintDate unique:", sorted(df["PrintDate"].dropna().dt.normalize().unique()))
print("nunique PrintDate:", df["PrintDate"].dt.normalize().nunique())

# 4) Year/Month
print(df.groupby(["Year", "Month"]).size())

# 5) Жёсткая проверка: всё ли в одну дату ответа?
d0 = df["AnswerTime"].dt.date.min()
print("all AnswerTime on same day?", (df["AnswerTime"].dt.date == d0).all())

In [ ]:
## Шаг D — идентификаторы клиента (заполненность)

In [ ]:
for col in ["UserContact", "PhoneFromLog", "ext_user_id", "ParticipateNumber"]:
    non_null = df[col].notna() & (df[col].astype(str).str.strip() != "") & (df[col].astype(str) != "None")
    print(col, "fill_rate=", round(non_null.mean(), 3), "nunique=", df.loc[non_null, col].nunique())

In [ ]:
## быстрая проверка ext_user_id (важно):

In [ ]:
print(df["ext_user_id"].value_counts().head(10))
print("share ext_user_id==0:", (df["ext_user_id"] == 0).mean().round(3))
print("nunique where ext_user_id!=0:", df.loc[df["ext_user_id"] != 0, "ext_user_id"].nunique())

In [ ]:
## Шаг E — вопросы и оценки

In [ ]:
print(df.groupby(["Question_ID", "Title"])["Answer_Value"].agg(["count", "mean", "min", "max"]))
print(df["Answer_Value"].value_counts().sort_index())

In [ ]:
##дополнение

In [ ]:
print(df.groupby(["Question_ID", "Title"], dropna=False)["Answer_Value"]
        .agg(count="count", mean="mean", min="min", max="max")
        .sort_values("count", ascending=False)
        .head(20))
print("Answer_Value value_counts:\n", df["Answer_Value"].value_counts(dropna=False).sort_index())

In [ ]:
## Шаг F — latency (секунды от заказа до ответа)

In [ ]:
lat = (pd.to_datetime(df["AnswerTime"]) - pd.to_datetime(df["PrintDateTime"])).dt.total_seconds()
print(lat.describe())

In [ ]:
## дополнение

In [ ]:
ans = df[df["Question_ID"].eq(10012) & df["Answer_Value"].notna()].copy()
print("answered rows:", len(ans))
lat = (pd.to_datetime(ans["AnswerTime"]) - pd.to_datetime(ans["PrintDateTime"])).dt.total_seconds()
print(lat.describe(percentiles=[0.5, 0.9, 0.95, 0.99]))
print("lat < 0 (answer before order):", (lat < 0).sum())
print("lat <= 60 sec:", (lat <= 60).sum(), "share:", round((lat <= 60).mean(), 3))
print("lat <= 300 sec:", (lat <= 300).sum(), "share:", round((lat <= 300).mean(), 3))

In [ ]:
## Шаг G — BlackList

In [ ]:
print(df["BlackList"].value_counts(dropna=False))

In [ ]:
## дополнение

In [ ]:
print(ans["BlackList"].value_counts(dropna=False))
# доля top-box среди blacklist vs нет (без печати контактов)
print(
    ans.assign(top=(ans["Answer_Value"] == 5))
       .groupby(ans["BlackList"].fillna("NA"), dropna=False)["top"]
       .agg(n="count", top_box_rate="mean")
)

In [ ]:
## Шаг H — baseline 5% (до фильтров) + черновик mapping

In [ ]:
# только рабочий вопрос + ответ
base = ans[ans["BlackList"].eq("לא")].copy()
def five_percent(s):
    return 100.0 * (s == 5).sum() / len(s)
print("overall 5% (BlackList=לא):", round(five_percent(base["Answer_Value"]), 2))
print("overall 5% (all answered):", round(five_percent(ans["Answer_Value"]), 2))
# store x month panel
panel = (base.groupby(["PrintStore", "Year", "Month"])["Answer_Value"]
           .agg(volume="count", five_pct=five_percent)
           .reset_index())
print(panel["five_pct"].describe())
print("panel rows:", len(panel))

In [ ]:
## И mapping:

In [ ]:
from fraud_guard.features import ColumnMapping
mapping = ColumnMapping(
    entity_key="UserContact",  # + fallback later
    store_id="PrintStore",
    event_ts="AnswerTime",
    survey_id="ParticipateNumber",
    question_id="Question_ID",
    answer_value="Answer_Value",
    channel="ContactType",
    checkout_ts="PrintDateTime",
)
mapping